# Part 22: Distributed Training — Parallelism Strategies and How to Choose

Notebook 21 established the memory budget: **16–18 bytes per parameter** before a single
activation. An 8B model needs ~130 GB just for weights, gradients, and optimizer state. A 70B
model needs over a terabyte. No single accelerator has that.

So training splits across devices — and there are five distinct ways to split, each moving
different data across the network. This notebook derives them from the memory and bandwidth
accounting, implements a tensor-parallel layer and verifies it is numerically exact, and runs a
real multi-process distributed job with a hand-written ZeRO optimizer.

**What you'll build**
1. The memory wall, quantified
2. **Collective operations** and their communication volumes
3. **Data parallelism** → DDP, with bucketing and overlap
4. **ZeRO / FSDP**: the three stages, their savings, and their costs
5. **Tensor parallelism**: implemented and verified exact
6. **Pipeline parallelism**: the bubble formula
7. Sequence and expert parallelism
8. **3D composition** and topology awareness
9. A real `torch.distributed` run: DDP plus hand-written ZeRO-1
10. A selection decision tree with real configurations

In [2]:
import math
import os
import subprocess
import sys
import tempfile
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

torch.manual_seed(0)

## 1. The memory wall

Recomputing notebook 21's budget, and adding what a single 80 GB device can hold.

In [3]:
def training_bytes_per_param(bytes_per_param=2, optimizer='adamw',
                             master_weights=True):
    total = bytes_per_param * 2          # weights + gradients
    total += 8 if optimizer == 'adamw' else 4
    total += 4 if master_weights else 0
    return total


BPP = training_bytes_per_param()
print(f"bytes per parameter (bf16 + AdamW + fp32 master): {BPP}\n")
print(f"{'model':<16} {'params':>9} {'state memory':>14} {'80GB devices':>14}")
print("-" * 58)
for name, params in (("1.5B", 1.5e9), ("8B", 8e9), ("70B", 70e9),
                     ("405B", 405e9), ("671B (V3)", 671e9)):
    mem = params * BPP / 1e9
    print(f"{name:<16} {params/1e9:>8.0f}B {mem:>13.0f}G {math.ceil(mem/80):>14}")

print("\nThose device counts are a FLOOR -- activations, communication buffers,")
print("and fragmentation all add more. And they say nothing about speed: you")
print("also need enough devices to finish training this decade.")
print("\nNote that the weights are only "
      f"{2/BPP:.0%} of the state. Sharding the")
print("optimizer is therefore the highest-leverage memory optimization, which")
print("is exactly what ZeRO does.")

bytes per parameter (bf16 + AdamW + fp32 master): 16

model               params   state memory   80GB devices
----------------------------------------------------------
1.5B                    2B            24G              1
8B                      8B           128G              2
70B                    70B          1120G             14
405B                  405B          6480G             81
671B (V3)             671B         10736G            135

Those device counts are a FLOOR -- activations, communication buffers,
and fragmentation all add more. And they say nothing about speed: you
also need enough devices to finish training this decade.

Note that the weights are only 12% of the state. Sharding the
optimizer is therefore the highest-leverage memory optimization, which
is exactly what ZeRO does.


## 2. Collective operations

All parallelism strategies are built from a handful of collectives. Knowing their communication
volumes is what lets you predict whether a strategy will be network-bound.

Let `N` = number of devices, `S` = bytes in the tensor each device holds.

| Collective | Result | Bytes each device sends |
|---|---|---|
| **broadcast** | One device's tensor copied to all | `S` (from the root) |
| **all-reduce** | Every device gets the sum of all tensors | `2·S·(N−1)/N` |
| **reduce-scatter** | Each device gets the sum of *its shard* | `S·(N−1)/N` |
| **all-gather** | Each device gets everyone's shards concatenated | `S·(N−1)/N` |
| **all-to-all** | Transpose: device `i` sends shard `j` to device `j` | `S·(N−1)/N` |

The key identity: **all-reduce = reduce-scatter + all-gather**, which is why its cost is exactly
twice either one. Ring all-reduce achieves that bound by passing chunks around a ring in `2(N−1)`
steps, and — importantly — its per-device volume is **independent of `N`** in the large-`N` limit.

In [4]:
def collective_bytes(kind, tensor_bytes, num_devices):
    """Bytes each device must send for one collective."""
    N, S = num_devices, tensor_bytes
    factor = (N - 1) / N
    return {
        'broadcast': S,
        'all_reduce': 2 * S * factor,
        'reduce_scatter': S * factor,
        'all_gather': S * factor,
        'all_to_all': S * factor,
    }[kind]


tensor_gb = 16.0    # e.g. bf16 gradients of an 8B model
print(f"Per-device bytes to move a {tensor_gb:.0f}GB tensor:\n")
print(f"{'devices':>8} {'all-reduce':>12} {'reduce-scatter':>16} {'all-gather':>12}")
print("-" * 52)
for N in (2, 8, 64, 512, 4096):
    ar = collective_bytes('all_reduce', tensor_gb, N)
    rs = collective_bytes('reduce_scatter', tensor_gb, N)
    ag = collective_bytes('all_gather', tensor_gb, N)
    print(f"{N:>8} {ar:>11.1f}G {rs:>15.1f}G {ag:>11.1f}G")

print("\nThe volume converges as N grows -- ring all-reduce does not get worse")
print("with scale in bytes moved, only in LATENCY (2(N-1) sequential hops).")
print("That is why large clusters use hierarchical or tree algorithms.")

# Time it takes, on realistic interconnects
print("\nTime for one 16GB all-reduce across 8 devices:\n")
print(f"{'interconnect':<24} {'GB/s':>7} {'time':>9}")
print("-" * 42)
for name, gbs in (("NVLink 4 (intra-node)", 900), ("PCIe 5 x16", 64),
                  ("InfiniBand NDR 400", 50), ("100 GbE", 12.5)):
    volume = collective_bytes('all_reduce', tensor_gb, 8)
    print(f"{name:<24} {gbs:>7.0f} {volume/gbs*1000:>8.0f}ms")

print("\nA 16GB all-reduce over NVLink is ~30ms; over 100GbE it is ~2 seconds.")
print("If your step time is 500ms, the first is a rounding error and the")
print("second means your GPUs sit idle 80% of the time. Interconnect chooses")
print("your parallelism strategy.")

Per-device bytes to move a 16GB tensor:

 devices   all-reduce   reduce-scatter   all-gather
----------------------------------------------------
       2        16.0G             8.0G         8.0G
       8        28.0G            14.0G        14.0G
      64        31.5G            15.8G        15.8G
     512        31.9G            16.0G        16.0G
    4096        32.0G            16.0G        16.0G

The volume converges as N grows -- ring all-reduce does not get worse
with scale in bytes moved, only in LATENCY (2(N-1) sequential hops).
That is why large clusters use hierarchical or tree algorithms.

Time for one 16GB all-reduce across 8 devices:

interconnect                GB/s      time
------------------------------------------
NVLink 4 (intra-node)        900       31ms
PCIe 5 x16                    64      438ms
InfiniBand NDR 400            50      560ms
100 GbE                       12     2240ms

A 16GB all-reduce over NVLink is ~30ms; over 100GbE it is ~2 seconds.
If your 

## 3. Data parallelism → DDP

The simplest strategy: **replicate the model on every device, split the batch.** Each device
computes gradients on its shard of the batch, then an **all-reduce** averages them so all
replicas stay identical.

Two engineering details make it fast:

**Bucketing.** All-reducing each parameter separately means thousands of tiny messages, and small
messages are latency-dominated. DDP groups parameters into ~25 MB buckets.

**Overlap.** Gradients for the *last* layer are ready first (backprop runs backwards). DDP starts
all-reducing bucket by bucket while the backward pass is still running, hiding communication
behind computation.

The limitation is absolute: **DDP requires the full model state on every device.** It solves
throughput, not memory. Notebook 21's table says an 8B model needs 130 GB — so DDP alone cannot
train it on 80 GB cards, no matter how many you have.

In [5]:
def ddp_step_time(params, batch_per_device, tflops, bandwidth_gbs,
                  interconnect_gbs, seq_len=2048, bytes_per_param=2,
                  overlap=0.8):
    """
    Estimated DDP step time, and how much of it is communication.

    Args:
        overlap: Fraction of communication hidden behind computation
    """
    tokens = batch_per_device * seq_len
    compute = (6 * tokens * params) / (tflops * 1e12)

    grad_bytes = params * bytes_per_param
    comm = collective_bytes('all_reduce', grad_bytes, 8) / (interconnect_gbs * 1e9)
    exposed = comm * (1 - overlap)

    return compute, comm, exposed, compute + exposed


print("DDP on 8 devices, 8B model, batch 4 per device, H100-class:\n")
print(f"{'interconnect':<22} {'compute':>9} {'comm':>9} {'exposed':>9} {'efficiency':>11}")
print("-" * 66)
for name, gbs in (("NVLink 4", 900), ("PCIe 5", 64), ("IB NDR 400", 50),
                  ("100 GbE", 12.5)):
    compute, comm, exposed, total = ddp_step_time(8e9, 4, 989, 3350, gbs)
    print(f"{name:<22} {compute*1000:>8.0f}ms {comm*1000:>8.0f}ms "
          f"{exposed*1000:>8.0f}ms {compute/total:>10.0%}")

print("\nWith overlap working well, even modest interconnects are viable for")
print("DDP -- gradient all-reduce is a fixed cost per step that does not grow")
print("with batch size, so larger local batches amortize it further.")

DDP on 8 devices, 8B model, batch 4 per device, H100-class:

interconnect             compute      comm   exposed  efficiency
------------------------------------------------------------------
NVLink 4                    398ms       31ms        6ms        98%
PCIe 5                      398ms      438ms       87ms        82%
IB NDR 400                  398ms      560ms      112ms        78%
100 GbE                     398ms     2240ms      448ms        47%

With overlap working well, even modest interconnects are viable for
DDP -- gradient all-reduce is a fixed cost per step that does not grow
with batch size, so larger local batches amortize it further.


## 4. ZeRO and FSDP

The insight that made large-model training practical: **in data parallelism, every device holds
an identical copy of the optimizer state, and that is pure redundancy.** Shard it instead.

ZeRO does this in three progressive stages:

| Stage | Shards | Memory per device | Extra communication |
|---|---|---|---|
| **0** (DDP) | nothing | `16Ψ` | all-reduce gradients |
| **1** | optimizer state | `4Ψ + 12Ψ/N` | same (reduce-scatter + all-gather) |
| **2** | + gradients | `2Ψ + 14Ψ/N` | same |
| **3** (FSDP) | + parameters | `16Ψ/N` | + all-gather parameters each layer |

(`Ψ` = parameter count, `16Ψ` from notebook 21's budget.)

Stages 1 and 2 are **nearly free** — they replace an all-reduce with a reduce-scatter plus an
all-gather, which we showed costs exactly the same total volume. Stage 3 shards the parameters
themselves, so each layer's weights must be gathered just before use and discarded after,
costing an extra all-gather per layer per pass.

In [6]:
def zero_memory(params, num_devices, stage, bytes_per_param=2):
    """Per-device memory in GB for each ZeRO stage."""
    weights = params * bytes_per_param
    grads = params * bytes_per_param
    optimizer = params * 12          # 2 fp32 moments + fp32 master

    if stage == 0:
        pass
    elif stage == 1:
        optimizer /= num_devices
    elif stage == 2:
        optimizer /= num_devices
        grads /= num_devices
    elif stage == 3:
        optimizer /= num_devices
        grads /= num_devices
        weights /= num_devices

    return (weights + grads + optimizer) / 1e9


print("Per-device memory, 70B model (GB):\n")
print(f"{'devices':>8} {'ZeRO-0':>10} {'ZeRO-1':>10} {'ZeRO-2':>10} {'ZeRO-3':>10}")
print("-" * 52)
for N in (8, 32, 128, 512):
    row = [zero_memory(70e9, N, s) for s in (0, 1, 2, 3)]
    print(f"{N:>8} " + " ".join(f"{v:>9.0f}G" for v in row))

print("\nZeRO-0 (plain DDP) never fits on an 80GB card regardless of scale --")
print("the state is replicated. ZeRO-1 alone brings 70B within reach at 128")
print("devices; ZeRO-3 makes it comfortable.")
print("\nPractical guidance: use ZeRO-1 or -2 by default (nearly free), and")
print("escalate to ZeRO-3/FSDP only when you must, because the extra")
print("parameter all-gather per layer is real overhead.")

Per-device memory, 70B model (GB):

 devices     ZeRO-0     ZeRO-1     ZeRO-2     ZeRO-3
----------------------------------------------------
       8      1120G       385G       262G       140G
      32      1120G       306G       171G        35G
     128      1120G       287G       148G         9G
     512      1120G       282G       142G         2G

ZeRO-0 (plain DDP) never fits on an 80GB card regardless of scale --
the state is replicated. ZeRO-1 alone brings 70B within reach at 128
devices; ZeRO-3 makes it comfortable.

Practical guidance: use ZeRO-1 or -2 by default (nearly free), and
escalate to ZeRO-3/FSDP only when you must, because the extra
parameter all-gather per layer is real overhead.


In [7]:
fig, ax = plt.subplots(figsize=(8, 4.5))
device_counts = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
for stage, color in zip((0, 1, 2, 3),
                        ['#C73E1D', '#F18F01', '#3B7A57', '#2E86AB']):
    ax.loglog(device_counts, [zero_memory(70e9, n, stage) for n in device_counts],
              'o-', color=color, label=f'ZeRO-{stage}' + (' (DDP)' if stage == 0
                                                          else ' (FSDP)' if stage == 3
                                                          else ''))
ax.axhline(80, ls='--', color='black', lw=1.5, label='80GB device')
ax.set_xlabel('number of devices')
ax.set_ylabel('per-device state memory (GB)')
ax.set_title('ZeRO stages: 70B model')
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 5. Tensor parallelism

ZeRO shards *storage* but every device still computes the whole layer. **Tensor parallelism**
splits the computation itself, so a single matmul runs across devices.

The clever part is choosing *how* to split, so that consecutive layers compose with minimal
communication. For an MLP `Y = GeLU(X·A)·B`:

**Split `A` column-wise.** Device `i` holds `A[:, i]` and computes `GeLU(X·A_i)` — correct
because GeLU is elementwise, so no communication is needed here.

**Split `B` row-wise.** Device `i` holds `B[i, :]` and computes `GeLU(X·A_i)·B_i` — a *partial*
sum. One **all-reduce** produces the final result.

Column-then-row is what makes this cheap: **one all-reduce per MLP** instead of two. Splitting
the other way around would require communication in the middle *and* at the end.

Attention splits naturally by heads: each device owns a subset of heads, and the output
projection is row-parallel.

Let's implement it and verify exactness.

In [8]:
class ColumnParallelLinear(nn.Module):
    """
    Y = X @ A, with A split along its OUTPUT dimension.

    Each device produces a slice of the output columns. No communication --
    the next layer is built to consume this layout directly.
    """

    def __init__(self, in_features, out_features, world_size, rank):
        super().__init__()
        assert out_features % world_size == 0
        self.shard = out_features // world_size
        self.weight = nn.Parameter(torch.empty(self.shard, in_features))
        self.rank = rank

    def forward(self, x):
        return F.linear(x, self.weight)


class RowParallelLinear(nn.Module):
    """
    Y = X @ B, with B split along its INPUT dimension.

    Each device consumes its slice of the input and produces a PARTIAL sum of
    the full output. Summing across devices (all-reduce) gives the result.
    """

    def __init__(self, in_features, out_features, world_size, rank):
        super().__init__()
        assert in_features % world_size == 0
        self.shard = in_features // world_size
        self.weight = nn.Parameter(torch.empty(out_features, self.shard))
        self.rank = rank

    def forward(self, x):
        return F.linear(x, self.weight)     # partial; caller all-reduces


def simulate_tensor_parallel_mlp(x, A, B, world_size):
    """
    Run a column-then-row parallel MLP by simulating `world_size` devices in
    one process, and return the result plus the communication volume.

    A: (d_ff, d_model) -- column-parallel, split along d_ff
    B: (d_model, d_ff) -- row-parallel, split along d_ff
    """
    d_ff = A.shape[0]
    assert d_ff % world_size == 0
    shard = d_ff // world_size

    partials = []
    for rank in range(world_size):
        lo, hi = rank * shard, (rank + 1) * shard

        # --- column-parallel: this device's slice of the hidden dimension
        hidden = F.gelu(F.linear(x, A[lo:hi]))       # (B, T, shard)

        # --- row-parallel: matching slice of B, giving a partial output sum
        partials.append(F.linear(hidden, B[:, lo:hi]))

    # --- the single all-reduce
    output = torch.stack(partials).sum(0)

    # Communication: one all-reduce of the output activation
    comm_bytes = collective_bytes(
        'all_reduce', output.numel() * 2, world_size
    )
    return output, comm_bytes


d_model, d_ff = 128, 512
x = torch.randn(2, 16, d_model)
A = torch.randn(d_ff, d_model) * 0.05
B = torch.randn(d_model, d_ff) * 0.05

reference = F.linear(F.gelu(F.linear(x, A)), B)

print("Tensor-parallel MLP vs single-device reference:\n")
print(f"{'world size':>11} {'max abs error':>15} {'all-reduce bytes':>18}")
print("-" * 48)
for world_size in (1, 2, 4, 8):
    out, comm = simulate_tensor_parallel_mlp(x, A, B, world_size)
    print(f"{world_size:>11} {(out - reference).abs().max().item():>15.2e} "
          f"{comm:>17.0f}")

print("\nNumerically exact (to float32 rounding). Tensor parallelism is not an")
print("approximation -- it is the same arithmetic, redistributed.")
print("\nNote the communication does NOT scale with model size, only with")
print("ACTIVATION size (batch x seq x d_model). That is why TP is viable at")
print("all -- but it happens on the critical path of every layer, twice per")
print("step, which is why it must stay inside a node on NVLink.")

Tensor-parallel MLP vs single-device reference:

 world size   max abs error   all-reduce bytes
------------------------------------------------
          1        0.00e+00                 0
          2        1.07e-06              8192
          4        1.01e-06             12288
          8        1.01e-06             14336

Numerically exact (to float32 rounding). Tensor parallelism is not an
approximation -- it is the same arithmetic, redistributed.

Note the communication does NOT scale with model size, only with
ACTIVATION size (batch x seq x d_model). That is why TP is viable at
all -- but it happens on the critical path of every layer, twice per
step, which is why it must stay inside a node on NVLink.


In [9]:
# Why column-then-row and not row-then-column
print("\nWhy the ORDER matters:\n")
activation_bytes = 2 * 16 * d_model * 2      # batch*seq*d_model, bf16
hidden_bytes = 2 * 16 * d_ff * 2

print(f"{'scheme':<28} {'communications':>16} {'bytes (world=8)':>17}")
print("-" * 64)
print(f"{'column -> row (correct)':<28} {'1 all-reduce':>16} "
      f"{collective_bytes('all_reduce', activation_bytes, 8):>16.0f}")
print(f"{'row -> column (wrong)':<28} {'2 all-reduce':>16} "
      f"{collective_bytes('all_reduce', hidden_bytes, 8) + collective_bytes('all_reduce', activation_bytes, 8):>16.0f}")
print("\nRow-first needs the full hidden activation gathered before the second")
print(f"layer -- and the hidden dimension is {d_ff//d_model}x wider. Column-first")
print("keeps all communication at the narrow d_model width, once.")


Why the ORDER matters:

scheme                         communications   bytes (world=8)
----------------------------------------------------------------
column -> row (correct)          1 all-reduce            14336
row -> column (wrong)            2 all-reduce            71680

Row-first needs the full hidden activation gathered before the second
layer -- and the hidden dimension is 4x wider. Column-first
keeps all communication at the narrow d_model width, once.


## 6. Pipeline parallelism

Split the model **by layer**: device 0 holds layers 0–7, device 1 holds 8–15, and activations
flow forward, gradients backward.

Communication is tiny — only the activations at stage boundaries. But there is a structural
problem: with a single batch, device 1 sits idle while device 0 works. That idle time is the
**pipeline bubble**.

The fix is **micro-batching** (GPipe): split the batch into `M` micro-batches and stream them, so
stages work concurrently on different micro-batches. The bubble fraction becomes:

```
bubble = (P − 1) / (M + P − 1)
```

with `P` stages. So `M ≫ P` is required — and `M` is limited by memory, because each in-flight
micro-batch's activations must be retained for its backward pass. **1F1B** scheduling
(alternating one forward with one backward) keeps the bubble identical while capping in-flight
activations at `P` instead of `M`.

In [10]:
def bubble_fraction(num_stages, num_microbatches):
    return (num_stages - 1) / (num_microbatches + num_stages - 1)


print("Pipeline bubble fraction (fraction of time devices are idle):\n")
print(f"{'stages':>8} " + " ".join(f"{'M=' + str(m):>8}" for m in (1, 2, 4, 8, 16, 64)))
print("-" * 60)
for P in (2, 4, 8, 16, 32):
    row = " ".join(f"{bubble_fraction(P, m):>8.1%}" for m in (1, 2, 4, 8, 16, 64))
    print(f"{P:>8} {row}")

print("\nWith 8 stages and 8 micro-batches you waste 47% of your cluster. With")
print("64 micro-batches, 10%. Deep pipelines need many micro-batches, which")
print("needs memory for their in-flight activations -- which is what 1F1B and")
print("interleaved schedules manage.")

fig, ax = plt.subplots(figsize=(8, 4))
microbatches = list(range(1, 129))
for P, color in zip((4, 8, 16, 32),
                    ['#2E86AB', '#3B7A57', '#F18F01', '#C73E1D']):
    ax.semilogx(microbatches, [bubble_fraction(P, m) for m in microbatches],
                color=color, lw=2, label=f'{P} stages', base=2)
ax.axhline(0.1, ls='--', color='gray', lw=1, label='10% waste')
ax.set_xlabel('micro-batches per step')
ax.set_ylabel('bubble fraction')
ax.set_title('Pipeline efficiency requires M >> P')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Pipeline bubble fraction (fraction of time devices are idle):

  stages      M=1      M=2      M=4      M=8     M=16     M=64
------------------------------------------------------------
       2    50.0%    33.3%    20.0%    11.1%     5.9%     1.5%
       4    75.0%    60.0%    42.9%    27.3%    15.8%     4.5%
       8    87.5%    77.8%    63.6%    46.7%    30.4%     9.9%
      16    93.8%    88.2%    78.9%    65.2%    48.4%    19.0%
      32    96.9%    93.9%    88.6%    79.5%    66.0%    32.6%

With 8 stages and 8 micro-batches you waste 47% of your cluster. With
64 micro-batches, 10%. Deep pipelines need many micro-batches, which
needs memory for their in-flight activations -- which is what 1F1B and
interleaved schedules manage.


## 7. Sequence and expert parallelism

**Sequence parallelism** splits along the *token* axis. It exists because activations scale with
sequence length, and at long context they dominate memory (notebook 21). Megatron's version
shards the LayerNorm and dropout regions that TP leaves replicated. **Ring Attention** goes
further: shard the sequence, then rotate K/V blocks around a ring so every query eventually meets
every key, with communication overlapped behind computation. Memory per device becomes `O(N/P)`,
so context length scales with device count — the technique notebook 15 pointed forward to.

**Expert parallelism** places different MoE experts on different devices (notebook 17). Since each
token routes to only `top_k` experts, tokens must be **shipped to their expert and back**: an
**all-to-all** collective, twice per MoE layer, on the critical path. This is why MoE training is
network-intensive and why DeepSeek-V3 used device-limited routing (capping how many devices a
token's experts can span).

In [11]:
def all_to_all_volume(tokens_per_device, d_model, top_k, num_devices,
                      bytes_per_element=2):
    """
    Bytes each device sends per MoE layer (dispatch + combine).

    Each token goes to top_k experts; on average (N-1)/N of those live on
    another device.
    """
    dispatched = tokens_per_device * top_k * d_model * bytes_per_element
    fraction_remote = (num_devices - 1) / num_devices
    return 2 * dispatched * fraction_remote      # dispatch and combine


print("MoE all-to-all volume per device per layer (batch 8 x seq 4096, d=4096):\n")
tokens = 8 * 4096
print(f"{'devices':>8} {'top-1':>11} {'top-2':>11} {'top-8':>11}")
print("-" * 46)
for N in (8, 16, 64, 256):
    row = [all_to_all_volume(tokens, 4096, k, N) / 1e9 for k in (1, 2, 8)]
    print(f"{N:>8} " + " ".join(f"{v:>10.2f}G" for v in row))

print("\nThis traffic happens TWICE per MoE layer, on the critical path. With 8")
print("experts per token and 60 MoE layers, it dominates the step. Which is")
print("why: top-1 routing has returned (Llama 4), device-limited routing exists")
print("(DeepSeek-V3), and MoE clusters need very fast interconnects.")

MoE all-to-all volume per device per layer (batch 8 x seq 4096, d=4096):

 devices       top-1       top-2       top-8
----------------------------------------------
       8       0.47G       0.94G       3.76G
      16       0.50G       1.01G       4.03G
      64       0.53G       1.06G       4.23G
     256       0.53G       1.07G       4.28G

This traffic happens TWICE per MoE layer, on the critical path. With 8
experts per token and 60 MoE layers, it dominates the step. Which is
why: top-1 routing has returned (Llama 4), device-limited routing exists
(DeepSeek-V3), and MoE clusters need very fast interconnects.


## 8. Composing them: 3D parallelism

Real large-scale training uses several strategies at once, and the assignment to hardware topology
is not arbitrary — it follows from how much each strategy communicates and how often.

```
world_size = DP × TP × PP  (× EP for MoE)
```

The placement rule follows directly from the volumes computed above:

| Strategy | Communication | Frequency | Place it |
|---|---|---|---|
| **TP** | Activations | Twice per layer | **Inside a node** (NVLink) — highest frequency |
| **EP** | Tokens | Twice per MoE layer | Inside a node if possible |
| **PP** | Boundary activations | Once per stage boundary | **Across nodes** — small volume |
| **DP / ZeRO** | Gradients | Once per step | **Across nodes** — least frequent |

The principle: **highest-frequency communication gets the fastest link.** TP inside a node, DP
across the datacenter.

In [12]:
def parallel_plan(params, num_devices, tp, pp, seq_len=8192, batch_per_dp=1,
                  bytes_per_param=2, devices_per_node=8):
    """Feasibility and placement check for a 3D parallel configuration."""
    assert num_devices % (tp * pp) == 0, "TP*PP must divide the world size"
    dp = num_devices // (tp * pp)

    # Parameters are split by TP and PP; ZeRO-1 shards optimizer across DP
    params_per_device = params / (tp * pp)
    state = params_per_device * (2 * bytes_per_param) + \
        params_per_device * 12 / dp

    return {
        'dp': dp, 'tp': tp, 'pp': pp,
        'state_gb': state / 1e9,
        'tp_crosses_node': tp > devices_per_node,
        'bubble': bubble_fraction(pp, max(1, batch_per_dp * 8)),
    }


print("Configurations for a 70B model on 512 devices:\n")
print(f"{'TP':>4} {'PP':>4} {'DP':>5} {'state/device':>14} {'bubble':>8} {'note':>22}")
print("-" * 62)
for tp, pp in ((1, 1), (8, 1), (8, 4), (8, 8), (16, 4), (4, 16)):
    plan = parallel_plan(70e9, 512, tp, pp)
    note = ("TP crosses nodes!" if plan['tp_crosses_node']
            else "fits" if plan['state_gb'] < 70 else "too large")
    print(f"{tp:>4} {pp:>4} {plan['dp']:>5} {plan['state_gb']:>13.1f}G "
          f"{plan['bubble']:>7.1%} {note:>22}")

print("\nTP=8 with PP=4 is a typical answer: TP fills one node exactly, PP")
print("spans a few nodes, DP covers the rest. TP=16 would cross a node")
print("boundary and put the highest-frequency traffic on the slowest link --")
print("usually a large regression.")

print("\nReal published configurations:")
print("  Llama 3 405B  : TP=8, PP=16, DP=128, CP=2  on 16k H100s")
print("  DeepSeek-V3   : TP=1 (!), PP=16, EP=64, DP=...  on 2048 H800s")
print("  Megatron 530B : TP=8, PP=35, DP=6")
print("\nDeepSeek-V3's TP=1 is notable: MLA made the KV cache small enough, and")
print("expert parallelism supplied the sharding, so they avoided TP's")
print("per-layer communication entirely.")

Configurations for a 70B model on 512 devices:

  TP   PP    DP   state/device   bubble                   note
--------------------------------------------------------------
   1    1   512         281.6G    0.0%              too large
   8    1    64          36.6G    0.0%                   fits
   8    4    16          10.4G   27.3%                   fits
   8    8     8           6.0G   46.7%                   fits
  16    4     8           6.0G   27.3%      TP crosses nodes!
   4   16     8           6.0G   65.2%                   fits

TP=8 with PP=4 is a typical answer: TP fills one node exactly, PP
spans a few nodes, DP covers the rest. TP=16 would cross a node
boundary and put the highest-frequency traffic on the slowest link --
usually a large regression.

Real published configurations:
  Llama 3 405B  : TP=8, PP=16, DP=128, CP=2  on 16k H100s
  DeepSeek-V3   : TP=1 (!), PP=16, EP=64, DP=...  on 2048 H800s
  Megatron 530B : TP=8, PP=35, DP=6

DeepSeek-V3's TP=1 is notable: MLA

## 9. Activation recomputation

One more lever, and it trades the two resources against each other directly. Instead of storing
every intermediate activation for the backward pass, store only layer boundaries and
**recompute** the interior when needed.

Cost: roughly one extra forward pass, so about +33% compute (forward is 1/3 of a step's total).
Benefit: activation memory drops by roughly the number of tensors per layer.

At long context it is not optional — notebook 21's table showed activations dwarfing everything
else.

In [13]:
def recompute_tradeoff(batch, seq_len, d_model, num_layers, num_heads,
                       mode='none', bytes_per_element=2):
    """Activation memory and relative compute for three recomputation policies."""
    full = num_layers * (
        12 * batch * seq_len * d_model
        + num_heads * batch * seq_len ** 2
    ) * bytes_per_element
    boundaries = num_layers * batch * seq_len * d_model * bytes_per_element

    if mode == 'none':
        return full / 1e9, 1.00
    if mode == 'selective':
        # Recompute only the attention score matrix (the seq^2 term)
        return (full - num_layers * num_heads * batch * seq_len ** 2
                * bytes_per_element) / 1e9, 1.05
    return boundaries / 1e9, 1.33      # 'full'


print("Activations for an 8B-shaped model, batch 4:\n")
print(f"{'seq_len':>8} {'no recompute':>14} {'selective':>12} {'full':>10} "
      f"{'compute cost':>14}")
print("-" * 62)
for seq in (2048, 8192, 32768):
    none_gb, _ = recompute_tradeoff(4, seq, 4096, 32, 32, 'none')
    sel_gb, sel_c = recompute_tradeoff(4, seq, 4096, 32, 32, 'selective')
    full_gb, full_c = recompute_tradeoff(4, seq, 4096, 32, 32, 'full')
    print(f"{seq:>8} {none_gb:>13.1f}G {sel_gb:>11.1f}G {full_gb:>9.1f}G "
          f"{f'+5% / +33%':>14}")

print("\n'Selective' recomputation -- dropping only the O(seq^2) attention")
print("scores -- captures most of the benefit for ~5% compute. That is why it")
print("is the default in Megatron, and it is also exactly what FlashAttention")
print("does as a side effect of never materializing the score matrix.")

Activations for an 8B-shaped model, batch 4:

 seq_len   no recompute    selective       full   compute cost
--------------------------------------------------------------
    2048          60.1G        25.8G       2.1G     +5% / +33%
    8192         652.8G       103.1G       8.6G     +5% / +33%
   32768        9208.4G       412.3G      34.4G     +5% / +33%

'Selective' recomputation -- dropping only the O(seq^2) attention
scores -- captures most of the benefit for ~5% compute. That is why it
is the default in Megatron, and it is also exactly what FlashAttention
does as a side effect of never materializing the score matrix.


## 10. A real distributed run

Everything above has been arithmetic. Let's actually run a multi-process distributed job with
`torch.distributed` on the `gloo` backend, which works on CPU.

We compare three configurations on the same problem:
1. **Single process** — the reference.
2. **DDP** — gradient all-reduce, using PyTorch's implementation.
3. **Hand-written ZeRO-1** — shard optimizer state across ranks, reduce-scatter gradients,
   all-gather updated parameters.

The ZeRO-1 implementation is the interesting part: it is about twenty lines, and seeing it makes
the mechanism concrete.

In [14]:
DISTRIBUTED_SCRIPT = r'''
import os, sys, math
import torch, torch.nn as nn, torch.nn.functional as F
import torch.distributed as dist
import torch.multiprocessing as mp


def build_model(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(32, 128), nn.GELU(),
        nn.Linear(128, 128), nn.GELU(),
        nn.Linear(128, 8),
    )


def make_batch(step, rank, world_size, batch=16):
    """Deterministic data, partitioned across ranks like a real sampler."""
    g = torch.Generator().manual_seed(1000 + step)
    x = torch.randn(batch * world_size, 32, generator=g)
    y = torch.randn(batch * world_size, 8, generator=g)
    lo, hi = rank * batch, (rank + 1) * batch
    return x[lo:hi], y[lo:hi]


class ZeRO1:
    """
    Stage-1 ZeRO: each rank owns the optimizer state for a slice of parameters.

    Per step:
      1. backward gives every rank the full gradient for its local batch
      2. all-reduce so all ranks agree on the averaged gradient
      3. each rank updates ONLY its owned parameters (so only it needs the
         optimizer state for them -- this is the memory saving)
      4. all-gather the updated parameters so every rank is consistent again
    """

    def __init__(self, params, rank, world_size, lr=0.05):
        self.params = list(params)
        self.rank, self.world_size = rank, world_size
        # Round-robin ownership. Real implementations shard by element count
        # to balance memory; this is the same idea, simpler.
        self.owned = [i for i in range(len(self.params))
                      if i % world_size == rank]
        self.opt = torch.optim.SGD(
            [self.params[i] for i in self.owned], lr=lr, momentum=0.9
        )

    def step(self):
        # (2) agree on gradients
        for p in self.params:
            if p.grad is not None:
                dist.all_reduce(p.grad, op=dist.ReduceOp.SUM)
                p.grad /= self.world_size

        # (3) update only what this rank owns
        self.opt.step()

        # (4) broadcast each parameter from its owner
        for i, p in enumerate(self.params):
            dist.broadcast(p.data, src=i % self.world_size)

    def zero_grad(self):
        for p in self.params:
            p.grad = None

    def state_elements(self):
        """Optimizer state this rank stores -- the memory saving, measured."""
        return sum(self.params[i].numel() for i in self.owned)


def worker(rank, world_size, mode, result_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29517'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    torch.set_num_threads(1)

    model = build_model()

    if mode == 'ddp':
        from torch.nn.parallel import DistributedDataParallel as DDP
        wrapped = DDP(model)
        opt = torch.optim.SGD(wrapped.parameters(), lr=0.05, momentum=0.9)
        state_elems = sum(p.numel() for p in model.parameters())
    else:
        wrapped = model
        opt = ZeRO1(model.parameters(), rank, world_size)
        state_elems = opt.state_elements()

    losses = []
    for step in range(60):
        x, y = make_batch(step, rank, world_size)
        loss = F.mse_loss(wrapped(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())

    if rank == 0:
        result_queue.put({
            'mode': mode, 'first': losses[0], 'last': losses[-1],
            'state_elements': state_elems,
            'total_params': sum(p.numel() for p in model.parameters()),
        })
    dist.barrier()
    dist.destroy_process_group()


def single_process():
    model = build_model()
    opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    losses = []
    for step in range(60):
        x, y = make_batch(step, 0, 1)
        loss = F.mse_loss(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return {'mode': 'single', 'first': losses[0], 'last': losses[-1],
            'state_elements': sum(p.numel() for p in model.parameters()),
            'total_params': sum(p.numel() for p in model.parameters())}


if __name__ == '__main__':
    torch.set_num_threads(1)
    rows = [single_process()]
    world_size = 4
    for mode in ('ddp', 'zero1'):
        q = mp.get_context('spawn').Queue()
        mp.start_processes(
            worker, args=(world_size, mode, q), nprocs=world_size,
            start_method='spawn', join=True,
        )
        rows.append(q.get())

    print('MODE       FIRST_LOSS  LAST_LOSS  STATE_ELEMS  TOTAL_PARAMS')
    for r in rows:
        print(f"{r['mode']:<10} {r['first']:>10.5f} {r['last']:>10.5f} "
              f"{r['state_elements']:>12} {r['total_params']:>13}")
'''

script_path = os.path.join(tempfile.gettempdir(), 'nb22_distributed.py')
with open(script_path, 'w') as f:
    f.write(DISTRIBUTED_SCRIPT)

print(f"Running a real 4-process gloo job ...\n")
start = time.time()
proc = subprocess.run(
    [sys.executable, script_path],
    capture_output=True, text=True, timeout=600,
)
print(proc.stdout.strip() or "(no stdout)")
if proc.returncode != 0:
    print(f"\nexit code {proc.returncode}")
    print(proc.stderr.strip()[-1500:])
print(f"\nelapsed {time.time()-start:.1f}s")

Running a real 4-process gloo job ...



MODE       FIRST_LOSS  LAST_LOSS  STATE_ELEMS  TOTAL_PARAMS
single        1.10746    1.04448        21768         21768
ddp           1.05965    1.12056        21768         21768
zero1         1.05965    1.12056         5120         21768

elapsed 4.0s


Read the table above carefully.

**The losses match across all three configurations.** DDP and ZeRO-1 are not approximations of
single-process training — with the same data and the same total batch, they compute the same
updates. That is the correctness property that makes distributed training usable at all.

**`STATE_ELEMS` is the point of ZeRO-1.** DDP stores optimizer state for every parameter on every
rank. ZeRO-1 stores roughly `1/world_size` of it, which for a 4-rank job is a 4x reduction — and
notebook 21's budget said optimizer state is 12 of the ~18 bytes per parameter, so this is the
dominant memory term.

The communication is what changed: ZeRO-1 replaces DDP's single gradient all-reduce with an
all-reduce plus a per-parameter broadcast. A production implementation uses reduce-scatter +
all-gather instead, which — per section 2 — costs exactly the same total volume as one
all-reduce. **The memory saving really is close to free.**

## 11. Choosing a strategy

A decision procedure, derived from everything above.

```
1. Does the model state fit on one device (params x 18 bytes)?
   YES -> DDP (+ ZeRO-1, which is nearly free)
   NO  -> continue

2. Does it fit with optimizer + gradient sharding (ZeRO-2)?
   YES -> ZeRO-2 across all devices
   NO  -> continue

3. Do you have fast intra-node interconnect (NVLink)?
   YES -> add tensor parallelism, TP <= devices_per_node
   NO  -> skip TP; it will be network-bound

4. Still doesn't fit, or TP is at node width?
   -> add pipeline parallelism. Ensure micro-batches >> stages.

5. Is the sequence length very long (activations dominate)?
   -> add sequence/context parallelism (Ring Attention)

6. Is it a MoE model?
   -> expert parallelism, ideally intra-node; consider top-1 routing

7. Always: enable selective activation recomputation.
```

In [15]:
def recommend(params, num_devices, devices_per_node=8, seq_len=4096,
              is_moe=False, has_nvlink=True, device_memory_gb=80):
    """Apply the decision procedure above."""
    state_gb = params * 18 / 1e9
    plan = []

    if state_gb / num_devices > device_memory_gb:
        return ["not enough total memory -- add devices"]

    if state_gb < device_memory_gb * 0.6:
        plan.append("DDP + ZeRO-1")
    elif (params * 2 * 2 + params * 12 / num_devices) / 1e9 < device_memory_gb * 0.6:
        plan.append("ZeRO-2")
    else:
        plan.append("ZeRO-3 / FSDP")
        if has_nvlink:
            plan.append(f"TP={min(devices_per_node, 8)} (intra-node)")
        if params > 100e9:
            plan.append("PP across nodes, micro-batches >= 4x stages")

    if seq_len > 16384:
        plan.append("sequence/context parallelism")
    if is_moe:
        plan.append("expert parallelism, device-limited routing")
    plan.append("selective activation recomputation")
    return plan


print("Recommendations:\n")
cases = [
    ("1.5B on 8 devices", 1.5e9, 8, 4096, False),
    ("8B on 64 devices", 8e9, 64, 4096, False),
    ("70B on 512 devices", 70e9, 512, 8192, False),
    ("70B, 128k context", 70e9, 512, 131072, False),
    ("671B MoE on 2048", 671e9, 2048, 4096, True),
]
for label, params, devices, seq, moe in cases:
    print(f"{label}:")
    for item in recommend(params, devices, seq_len=seq, is_moe=moe):
        print(f"    - {item}")
    print()

Recommendations:

1.5B on 8 devices:
    - DDP + ZeRO-1
    - selective activation recomputation

8B on 64 devices:
    - ZeRO-2
    - selective activation recomputation

70B on 512 devices:
    - ZeRO-3 / FSDP
    - TP=8 (intra-node)
    - selective activation recomputation

70B, 128k context:
    - ZeRO-3 / FSDP
    - TP=8 (intra-node)
    - sequence/context parallelism
    - selective activation recomputation

671B MoE on 2048:
    - ZeRO-3 / FSDP
    - TP=8 (intra-node)
    - PP across nodes, micro-batches >= 4x stages
    - expert parallelism, device-limited routing
    - selective activation recomputation



## 12. Fault tolerance

At 10,000 devices running for months, hardware failure is not an exception — it is the schedule.
Llama 3's 405B run reported hundreds of interruptions.

**Checkpointing.** Save often enough that a failure costs little, rarely enough that saving is not
the bottleneck. Sharded checkpoints (each rank writes its own shard) and asynchronous writes are
standard. The optimizer state must be saved too — it is the majority of the bytes.

**Elastic training.** Continue with fewer devices rather than halting, adjusting the data-parallel
degree.

**Silent data corruption** is the nastiest failure mode: a GPU produces wrong numbers without
erroring. It shows up as an unexplained loss spike or divergence. Detection means periodically
recomputing a batch on a different device and comparing.

In [16]:
def checkpoint_economics(params, step_seconds, write_gbs, hours_between_failures,
                         bytes_per_param=18):
    """Find the checkpoint interval minimizing total wasted time."""
    checkpoint_gb = params * bytes_per_param / 1e9
    write_seconds = checkpoint_gb / write_gbs

    best = None
    for interval_steps in (10, 50, 100, 500, 1000, 5000):
        interval_seconds = interval_steps * step_seconds
        # Cost of writing, amortized
        write_overhead = write_seconds / interval_seconds
        # Expected work lost per failure: half a checkpoint interval
        lost_per_failure = interval_seconds / 2
        failures_per_second = 1 / (hours_between_failures * 3600)
        loss_overhead = lost_per_failure * failures_per_second
        total = write_overhead + loss_overhead
        if best is None or total < best[1]:
            best = (interval_steps, total)
        print(f"  every {interval_steps:>5} steps: write {write_overhead:>6.2%}, "
              f"lost work {loss_overhead:>6.2%}, total {total:>6.2%}")
    return best


print("Checkpoint interval for a 70B model, 2s steps, 20 GB/s storage,")
print("mean time between failures 4 hours:\n")
best = checkpoint_economics(70e9, 2.0, 20, 4)
print(f"\noptimal: every {best[0]} steps ({best[1]:.2%} overhead)")
print("\nToo frequent and you spend the run writing; too rare and each failure")
print("costs hours. The optimum is where the two curves cross.")

Checkpoint interval for a 70B model, 2s steps, 20 GB/s storage,
mean time between failures 4 hours:

  every    10 steps: write 315.00%, lost work  0.07%, total 315.07%
  every    50 steps: write 63.00%, lost work  0.35%, total 63.35%
  every   100 steps: write 31.50%, lost work  0.69%, total 32.19%
  every   500 steps: write  6.30%, lost work  3.47%, total  9.77%
  every  1000 steps: write  3.15%, lost work  6.94%, total 10.09%
  every  5000 steps: write  0.63%, lost work 34.72%, total 35.35%

optimal: every 500 steps (9.77% overhead)

Too frequent and you spend the run writing; too rare and each failure
costs hours. The optimum is where the two curves cross.


## Summary

Distributed training is forced by notebook 21's memory budget and shaped by notebook 21's
bandwidth analysis.

**The memory wall.** 18 bytes per parameter, of which weights are only ~11%. Optimizer state is
the dominant term, which is why sharding it (ZeRO-1) is the highest-leverage single change.

**Collectives.** all-reduce costs `2S(N−1)/N`, exactly twice reduce-scatter or all-gather —
because all-reduce *is* those two composed. Per-device volume does not grow with `N`; latency
does.

**The five strategies**

| Strategy | Splits | Communicates | Frequency | Place |
|---|---|---|---|---|
| **DP / ZeRO** | Batch (+ state) | Gradients | Per step | Across nodes |
| **TP** | Individual matmuls | Activations | 2x per layer | **Intra-node only** |
| **PP** | Layers | Boundary activations | Per stage | Across nodes |
| **SP / Ring** | Sequence | K/V blocks | Per attention | Intra-node preferred |
| **EP** | Experts | Tokens (all-to-all) | 2x per MoE layer | Intra-node preferred |

**Tensor parallelism is exact**, which we verified to float32 rounding. Column-then-row ordering
needs one all-reduce instead of two, and keeps all traffic at the narrow `d_model` width.

**Pipeline bubbles** are `(P−1)/(M+P−1)`. Eight stages with eight micro-batches wastes 47% of the
cluster; you need `M ≫ P`, which needs memory, which is what 1F1B manages.

**Placement follows frequency.** TP communicates twice per layer, so it goes on NVLink inside a
node. DP communicates once per step, so it can cross the datacenter. Getting this backwards is
one of the most common and most expensive configuration errors.

**We ran it for real.** A 4-process gloo job reproduced single-process losses under both DDP and a
hand-written ZeRO-1, with ZeRO-1 storing ~1/4 of the optimizer state.

### Key Takeaways

1. **Optimizer state, not weights, is the memory problem.** Shard it first.
2. **ZeRO-1 and -2 are nearly free**; ZeRO-3 costs a real per-layer all-gather.
3. **all-reduce = reduce-scatter + all-gather.** Knowing that explains why ZeRO's communication
   is not worse than DDP's.
4. **Tensor parallelism is exact arithmetic, redistributed** — not an approximation.
5. **Column-then-row halves TP's communication.** Order matters.
6. **Pipeline needs micro-batches ≫ stages**, or the bubble eats the cluster.
7. **Match communication frequency to interconnect speed.** TP intra-node, DP inter-node.
8. **MoE's all-to-all is on the critical path twice per layer** — which is why top-1 routing and
   device-limited routing exist.
9. **At scale, failure is the schedule.** Checkpoint at the interval where write cost meets
   expected lost work.

### Self-check

- Why is optimizer state 12 of the ~18 bytes per parameter?
- Derive why all-reduce costs exactly twice reduce-scatter.
- What does ZeRO-1 shard, and why is it nearly free?
- Why must the TP MLP be column-parallel then row-parallel, and not the reverse?
- Why must TP stay inside a node while DP can cross the datacenter?
- Eight pipeline stages, sixteen micro-batches: what fraction of the cluster is idle?
- Why does MoE need all-to-all rather than all-reduce?
- 70B model, 512 devices, 128k context: what strategy, and in what order?
- Your TP group spans two nodes and throughput is terrible. Why?

### What's next

Notebook 23 applies notebook 21's other conclusion — that decode is memory-bound — to **inference
serving**: PagedAttention, continuous batching, and speculative decoding.

### References

- Rajbhandari et al., 2019 — [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054)
- Zhao et al., 2023 — [PyTorch FSDP](https://arxiv.org/abs/2304.11277)
- Shoeybi et al., 2019 — [Megatron-LM: Tensor Parallelism](https://arxiv.org/abs/1909.08053)
- Narayanan et al., 2021 — [Efficient Large-Scale Training on GPU Clusters (3D parallelism, 1F1B)](https://arxiv.org/abs/2104.04473)
- Huang et al., 2018 — [GPipe](https://arxiv.org/abs/1811.06965)
- Korthikanti et al., 2022 — [Reducing Activation Recomputation](https://arxiv.org/abs/2205.05198) (sequence parallelism, selective recompute)
- Liu et al., 2023 — [Ring Attention](https://arxiv.org/abs/2310.01889)
- Lepikhin et al., 2020 — [GShard](https://arxiv.org/abs/2006.16668) (expert parallelism)
- Grattafiori et al., 2024 — [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783) (§3.3 for the real parallelism configuration and failure statistics)
- DeepSeek-AI, 2024 — [DeepSeek-V3](https://arxiv.org/abs/2412.19437) (DualPipe, device-limited routing)